![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

https://www.quantconnect.com/datasets/algoseek-us-futures

### Get Data

In [1]:
from QuantConnect import *
from QuantConnect.Research import QuantBook
from datetime import datetime
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

qb = QuantBook()

# Date range
start = pd.Timestamp(2022, 5, 1, tz="UTC") 
end = pd.Timestamp(2023, 5, 1, tz="UTC")  

# --- Commodity Futures (already selected) ---
commodity_futures = {
    # Metals
    'GC': Futures.Metals.GOLD,
    'SI': Futures.Metals.SILVER,
    'HG': Futures.Metals.COPPER,
    
    # Energy
    'CL': Futures.Energy.CRUDE_OIL_WTI,
    'NG': Futures.Energy.NATURAL_GAS,
        
}

# --- Currencies Futures (Top liquid FX pairs & crypto-linked) ---
currency_futures = {
    #'6E': Futures.Currencies.EUR,      # Euro FX (EUR/USD)
    #'6B': Futures.Currencies.GBP,      # British Pound (GBP/USD)
    #'6J': Futures.Currencies.JPY,      # Japanese Yen (USD/JPY)      
    #'6A': Futures.Currencies.AUD,      # Australian Dollar (AUD/USD)
    #'6S': Futures.Currencies.CHF,      # Swiss Franc (USD/CHF)
    #'6M': Futures.Currencies.MXN,      # Mexican Peso (USD/MXN)
}

# --- Financial Futures (Interest rates / Bonds) ---
financial_futures = {
    'ZT': Futures.Financials.Y_2_TREASURY_NOTE,   # 2Y US Treasury
    'GE': Futures.Financials.EURO_DOLLAR,         # EuroDollar Futures (short-term rates)
}

# --- Index Futures (Macro sentiment / risk-on proxies) ---
index_futures = {
    'ES': Futures.Indices.SP_500_E_MINI,          # S&P 500 E-mini
    'YM': Futures.Indices.DOW_30_E_MINI          # Dow Jones Industrial                
}

# Merge all futures into one dictionary
all_futures = {}
all_futures.update(commodity_futures)
all_futures.update(currency_futures)
all_futures.update(financial_futures)
all_futures.update(index_futures)

# Add futures and get data
dfs_by_symbol = {}

for symbol_name, future_enum in all_futures.items():
    try:
        print(f"Fetching {symbol_name}...")
        
        # Add future
        future = qb.add_future(future_enum)
        
        # Get minute-level OHLCV data
        history = qb.history(
            future.symbol,
            start,
            end,
            Resolution.MINUTE,
            fill_forward=True,
            data_mapping_mode=DataMappingMode.OPEN_INTEREST,
            contract_depth_offset=0
        )
        
        # Get minute-level open interest data
        oi_history = qb.history(
            OpenInterest,
            future.symbol,
            start,
            end,
            Resolution.MINUTE,
            fill_forward=True,
            data_mapping_mode=DataMappingMode.OPEN_INTEREST,
            contract_depth_offset=0
        )
        
        if history.empty:
            print(f"  No data for {symbol_name}")
            continue
        
        # Reset index to access columns
        history = history.reset_index()
        
        # Add open interest if available
        if not oi_history.empty:
            oi_history = oi_history.reset_index()
            # Merge on time
            history = history.merge(
                oi_history[['time', 'openinterest']], 
                on='time', 
                how='left'
            )
        
        # Create 5-minute groups
        history['time_5min'] = history['time'].dt.floor('5min')
        
        # Resample to 5-minute bars
        agg_dict = {
            'open': 'first',
            'high': 'max',
            'low': 'min',
            'close': 'last',
            'volume': 'sum'
        }
        
        # Add open interest aggregation if column exists
        if 'openinterest' in history.columns:
            agg_dict['openinterest'] = 'last'  # Take last value in 5-min window
        
        history_5min = history.groupby('time_5min').agg(agg_dict)
        
        # Set index name
        history_5min.index.name = 'time'
        
        # Store
        dfs_by_symbol[symbol_name] = history_5min
        
        oi_info = f", OI: {'✓' if 'openinterest' in history_5min.columns else '✗'}"
        print(f"  {symbol_name}: {len(history_5min)} bars, "
              f"{history_5min.index.min()} to {history_5min.index.max()}{oi_info}")
        
    except Exception as e:
        print(f"  Error with {symbol_name}: {str(e)}")

print(f"\n✓ Successfully loaded {len(dfs_by_symbol)} commodity futures")
print(f"Symbols: {list(dfs_by_symbol.keys())}")

# Verify data quality
print("\nData Quality Check:")
for symbol, df in dfs_by_symbol.items():
    zero_vol = (df['volume'] == 0).sum()
    pct_zero = zero_vol / len(df) * 100
    print(f"{symbol}: {len(df)} bars, {pct_zero:.1f}% zero volume")

In [2]:
print(dfs_by_symbol)


### Add Features

In [3]:
import pandas as pd
import numpy as np
from scipy import stats

def add_futures_features(df, ticker):
    """
    Stationary features for commodity futures as BTC predictors.
    All features prefixed with ticker_futures_. No raw OHLCV in output.
    """
    features = df.copy()
    
    # ============================================================
    # PRICE MOMENTUM (Returns - stationary)
    # ============================================================
    features['ret_1'] = features['close'].pct_change(1)      # 5-min
    features['ret_3'] = features['close'].pct_change(3)      # 15-min
    features['ret_12'] = features['close'].pct_change(12)    # 1-hour
    features['ret_48'] = features['close'].pct_change(48)    # 4-hour
    features['ret_288'] = features['close'].pct_change(288)  # 1-day
    
    # Momentum percentile ranks
    features['ret_pctrank_12'] = features['ret_12'].rolling(48).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    features['ret_pctrank_48'] = features['ret_48'].rolling(288).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # ============================================================
    # PRICE SEASONALITY (vs previous day)
    # ============================================================
    features['ret_vs_1d_ago'] = features['close'].pct_change(288) / (features['ret_288'].rolling(288).std() + 1e-8)
    features['price_pctrank_1d'] = features['close'].rolling(288).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # ============================================================
    # RELATIVE VOLUME (Ratios - stationary)
    # ============================================================
    vma_12 = features['volume'].rolling(12).mean()
    vma_48 = features['volume'].rolling(48).mean()
    vma_288 = features['volume'].rolling(288).mean()
    
    features['rvol_1h'] = features['volume'] / (vma_12 + 1e-8)
    features['rvol_4h'] = features['volume'] / (vma_48 + 1e-8)
    features['rvol_1d'] = features['volume'] / (vma_288 + 1e-8)
    
    features['vol_pctrank_48'] = features['volume'].rolling(48).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # ============================================================
    # VOLUME SEASONALITY (vs previous day)
    # ============================================================
    features['vol_vs_1d_ago'] = features['volume'] / (features['volume'].shift(288) + 1e-8)
    features['vol_vs_1d_avg'] = features['volume'] / (vma_288 + 1e-8)
    
    # ============================================================
    # VOLATILITY (Normalized - stationary)
    # ============================================================
    features['rvol_12'] = features['ret_1'].rolling(12).std()   # 1-hour
    features['rvol_48'] = features['ret_1'].rolling(48).std()   # 4-hour
    features['rvol_288'] = features['ret_1'].rolling(288).std() # 1-day
    
    # ATR (normalized by price)
    tr1 = features['high'] - features['low']
    tr2 = np.abs(features['high'] - features['close'].shift(1))
    tr3 = np.abs(features['low'] - features['close'].shift(1))
    true_range = np.maximum(tr1, np.maximum(tr2, tr3))
    
    atr_12 = true_range.rolling(12).mean()
    atr_48 = true_range.rolling(48).mean()
    
    features['atr_12_pct'] = atr_12 / features['close']
    features['atr_48_pct'] = atr_48 / features['close']
    
    features['vol_pctrank_48_rvol'] = features['rvol_48'].rolling(288).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # ============================================================
    # VOLATILITY SEASONALITY (vs previous day)
    # ============================================================
    features['rvol_vs_1d_ago'] = features['rvol_48'] / (features['rvol_48'].shift(288) + 1e-8)
    features['vol_regime_change'] = (features['rvol_48'] - features['rvol_48'].shift(288)) / (features['rvol_288'] + 1e-8)
    
    # ============================================================
    # OPEN INTEREST (Changes & Trends - stationary)
    # ============================================================
    if 'openinterest' in features.columns:
        features['oi_ret_12'] = features['openinterest'].pct_change(12)
        features['oi_ret_48'] = features['openinterest'].pct_change(48)
        features['oi_ret_288'] = features['openinterest'].pct_change(288)
        
        features['oi_trend_48'] = (features['openinterest'] - features['openinterest'].shift(48)) / (features['openinterest'].rolling(48).std() + 1e-8)
        features['oi_trend_288'] = (features['openinterest'] - features['openinterest'].shift(288)) / (features['openinterest'].rolling(288).std() + 1e-8)
        
        features['vol_oi_ratio'] = features['volume'].rolling(12).sum() / (features['openinterest'] + 1e-8)
        features['vol_oi_ratio_48'] = features['volume'].rolling(48).sum() / (features['openinterest'] + 1e-8)
        
        price_chg_48 = features['close'].pct_change(48)
        oi_chg_48 = features['openinterest'].pct_change(48)
        features['price_oi_divergence'] = (price_chg_48 - oi_chg_48).abs()
        
        features['oi_pctrank_288'] = features['openinterest'].rolling(288).apply(
            lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
        )
    
    # ============================================================
    # DROP RAW OHLCV + OI (keep only derived features)
    # ============================================================
    features = features.drop(['open', 'high', 'low', 'close', 'volume', 'openinterest'], 
                             axis=1, errors='ignore')
    
    # ============================================================
    # PREFIX ALL COLUMNS WITH TICKER_FUTURES_
    # ============================================================
    rename_dict = {col: f'{ticker.lower()}_futures_{col}' for col in features.columns}
    features = features.rename(columns=rename_dict)
    
    return features


# Apply to all futures
print("="*60)
print("CREATING FUTURES FEATURES (STATIONARY ONLY)")
print("="*60)

futures_features = {}

for sym in dfs_by_symbol.keys():
    print(f"\n{sym}...", end=" ")
    features = add_futures_features(dfs_by_symbol[sym].copy(), sym)
    futures_features[sym] = features
    print(f"✓ {features.shape[1]} features, {features.shape[0]} rows")

### Merge Data

In [4]:
import pandas as pd

def merge_symbol_dataframes(dfs_by_symbol):
    """
    Merges multiple symbol DataFrames into a single wide DataFrame.
    Assumes features are already prefixed - no additional prefixing.
    """
    merged_df = None
    
    for symbol, df in dfs_by_symbol.items():
        # Get symbol name as string
        symbol_name = symbol if isinstance(symbol, str) else str(symbol)
        
        # Make a copy
        df_copy = df.copy()
        
        # Ensure time is the index
        if 'time' in df_copy.columns:
            df_copy = df_copy.set_index('time')
        
        # Merge
        if merged_df is None:
            merged_df = df_copy
        else:
            merged_df = merged_df.join(df_copy, how='outer')
    
    # Sort index
    merged_df = merged_df.sort_index()
    
    print(f"✓ Merged {len(dfs_by_symbol)} symbols")
    print(f"  Shape: {merged_df.shape}")
    print(f"  Date range: {merged_df.index.min()} to {merged_df.index.max()}")
    
    return merged_df


# Execute merge
print("="*60)
print("MERGING ALL FUTURES")
print("="*60)
merged_futures = merge_symbol_dataframes(futures_features)

print(f"\nSample columns: {merged_futures.columns.tolist()[:5]}")

## HANDLING NANS


In [5]:
import pandas as pd
import numpy as np

print("="*60)
print("MERGED FUTURES - DATA QUALITY ASSESSMENT")
print("="*60)

# === STEP 1: INITIAL ASSESSMENT ===
print(f"\n1. INITIAL STATE")
print(f"   Shape: {merged_futures.shape}")
print(f"   Date range: {merged_futures.index.min()} to {merged_futures.index.max()}")
print(f"   Duration: {(merged_futures.index.max() - merged_futures.index.min()).days} days")

# === STEP 2: CHECK FOR BROKEN FEATURES (>90% NaN) ===
print(f"\n2. BROKEN FEATURES CHECK (>90% NaN)")
nan_pct = (merged_futures.isnull().sum() / len(merged_futures)) * 100
broken = nan_pct[nan_pct > 90]

if len(broken) > 0:
    print(f"   ⚠️  Found {len(broken)} broken features:")
    for feat, pct in broken.items():
        print(f"      {feat}: {pct:.1f}% NaN")
else:
    print(f"   ✓ No broken features")

# === STEP 3: NaN DISTRIBUTION ===
print(f"\n3. NaN DISTRIBUTION")
nan_counts = merged_futures.isnull().sum().sort_values(ascending=False)
has_nans = nan_counts[nan_counts > 0]

if len(has_nans) > 0:
    print(f"   Features with NaNs: {len(has_nans)} / {len(merged_futures.columns)}")
    print(f"   Total NaNs: {nan_counts.sum():,}")
    print(f"\n   Top 10:")
    for feat, count in has_nans.head(10).items():
        pct = (count / len(merged_futures)) * 100
        print(f"      {feat}: {count} ({pct:.1f}%)")
else:
    print(f"   ✓ No NaNs")

# === STEP 4: DETERMINE WARMUP PERIOD ===
print(f"\n4. WARMUP PERIOD ANALYSIS")

nan_by_row = merged_futures.isnull().sum(axis=1)
first_complete_row_idx = (nan_by_row == 0).idxmax() if (nan_by_row == 0).any() else None

if first_complete_row_idx:
    warmup_rows = merged_futures.index.get_loc(first_complete_row_idx)
    print(f"   First complete row: index {warmup_rows} ({first_complete_row_idx})")
    print(f"   Recommended warmup: {warmup_rows} rows")
else:
    print(f"   ⚠️  No complete rows found")
    min_nans_idx = nan_by_row.idxmin()
    warmup_rows = merged_futures.index.get_loc(min_nans_idx)
    print(f"   Best row has {nan_by_row[min_nans_idx]} NaNs at index {warmup_rows}")
    print(f"   Using {warmup_rows} as warmup period")

# === STEP 5: NaN PATTERN ANALYSIS ===
print(f"\n5. NaN PATTERN ANALYSIS")

only_start_nans = True
middle_nan_features = []

for col in has_nans.index[:10]:  # Check top 10
    first_valid = merged_futures[col].first_valid_index()
    if first_valid:
        first_idx = merged_futures.index.get_loc(first_valid)
        middle_nans = merged_futures[col].iloc[first_idx:].isnull().sum()
        if middle_nans > 0:
            only_start_nans = False
            middle_nan_features.append((col, middle_nans))

if only_start_nans:
    print(f"   ✓ NaNs only at start (warmup)")
else:
    print(f"   ⚠️  NaNs in middle of data (likely market hours/closures)")
    print(f"   Examples:")
    for feat, count in middle_nan_features[:5]:
        print(f"      {feat}: {count} middle NaNs")

# === STEP 6: DECISION ===
print(f"\n{'='*60}")
print("CLEANING STRATEGY")
print('='*60)
print(f"1. DROP {len(broken)} broken features (>90% NaN)")
print(f"2. DROP first {warmup_rows} rows (warmup period)")
print(f"3. KEEP remaining NaNs (market closed = information)")
print(f"4. Compatible with: XGBoost, LightGBM, CatBoost")

# === STEP 7: EXECUTE CLEANING ===
print(f"\n{'='*60}")
print("EXECUTING CLEANING")
print('='*60)

# Start with copy
merged_futures_clean = merged_futures.copy()

# Drop broken features
if len(broken) > 0:
    merged_futures_clean = merged_futures_clean.drop(columns=broken.index)
    print(f"✓ Dropped {len(broken)} broken features")

# Drop warmup period
print(f"Before: {merged_futures_clean.shape}")
merged_futures_clean = merged_futures_clean.iloc[warmup_rows:].copy()
print(f"After warmup drop: {merged_futures_clean.shape}")

# Report final NaN count (NOT filling them)
final_nans = merged_futures_clean.isnull().sum().sum()
print(f"Preserved NaNs: {final_nans:,}")

# === FINAL STATE ===
print(f"\n{'='*60}")
print("FINAL DATASET")
print('='*60)
print(f"Shape: {merged_futures_clean.shape}")
print(f"Date range: {merged_futures_clean.index.min()} to {merged_futures_clean.index.max()}")
print(f"Duration: {(merged_futures_clean.index.max() - merged_futures_clean.index.min()).days} days")
print(f"NaNs: {final_nans:,} ({final_nans / merged_futures_clean.size * 100:.2f}% of total values)")

print(f"\n✓✓✓ MERGED FUTURES READY FOR TREE MODELS ✓✓✓")
print(f"    NaNs represent market closures - this is valid information")

print(f"\nSample (first 3 rows, first 8 columns):")
print(merged_futures_clean.iloc[:3, :8])


In [6]:
# Save
print(f"\n{'='*60}")
print("SAVING")
print('='*60)
merged_futures_clean.to_parquet('commodities_clean.parquet')
print(f"✓ Saved: commodities_clean.parquet")